# Suite CNT — Containers

A container holds elements by index (`Schemas.OfContainer`): a property's value, read and written through
`Visitors.OfContainer`, its elements each through a `Visitors.OfAny`. Dense, its elements are a packed array; sparse,
blocks of them; with neither, it is dense for an integer-like index and a dict for any other hashable one. These cases
cover proxies, iterators, the value objects containers hold, and snapshots.

In [ ]:
import { Builders as B, Comparison, JSON, Plain, Proxies, Schemas as S, Validators, YAML } from "@mbse/schemas/Framework";
import { DecodeError, ReadOnlyError, ValueError } from "@mbse/schemas/Framework/Errors";
import { repr } from "@mbse/schemas/Framework/Repr";
import { assert, document, equal, listed, map, raises, ref, root_of, same, single } from "./support.js";

/** Cell, a value object of two ints; Tags, a dense container of text; Grid, of floats by cell; and Card, holding a
 * container of each layout. */
function install(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("cnt.cards/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Cell: S.OfObject, Tags: S.OfContainer, Grid: S.OfContainer, Card: S.OfObject })
    .spec([
      (_: any) => _.Cell.name("Cell").properties({ row: S.OfInt, col: S.OfInt }),
      (_: any) => _.Tags.name("Tags").index([S.OfInt]).element([S.OfStr]), // dense
      (_: any) => _.Grid.name("Grid").index([_.Cell]).element([S.OfFloat]), // dense, of two indices
      (_: any) => _.Card.name("Card").ref(true).properties({
        tags: _.Tags,
        phones: S.OfContainer((c: any) => c.index([S.OfStr]).element([S.OfObject((o: any) => o.properties({
          number: S.OfStr, label: S.OfStr }))])), // a dict
        grid: _.Grid,
        sparse: S.OfContainer((c: any) => c.index([S.OfInt]).element([S.OfStr]).layout("sparse")),
        rows: S.OfContainer((c: any) => c.index([S.OfInt]).element([_.Tags])), // nested
        name: S.OfStr,
      }),
    ])]))]);
}

const store: any = new B.Store();
const Int = store.Schemas.OfInt.as_schema(); // the store's own
const Text = store.Schemas.OfStr.as_schema(); // the store's own
const Float = store.Schemas.OfFloat.as_schema(); // the store's own
store.transaction(install);
const [Cell, Tags, Grid, Card] = ["Cell", "Tags", "Grid", "Card"].map((name) => store.registered(name));

/** The container a property holds, as its visitor; null where it holds none (its visitor then visits nothing). */
function container(read: any): any {
  const found: any[] = [];
  read.as_value((a: any) => a.as_container((c: any) => found.push(c) && c));
  return found[0].as_schema() !== null ? found[0] : null;
}

/** An element's scalar, or null. */
function valueOfAny(e: any): unknown {
  const found: unknown[] = [];
  e.as_scalar((s: any) => found.push(s.get()) && s);
  return found.length > 0 ? found[0] : null;
}

/** A container's elements present, each with its index, as scalars. */
function scalars(c: any): unknown[] {
  const out: unknown[] = [];
  c.as_entries((pairs: Iterable<[unknown, any]>) => {
    for (const [index, e] of pairs) out.push([index, valueOfAny(e)]);
  });
  return out;
}

/** Each position of a dense or sparse container, with its element's scalar, by its iterator. */
function walk(c: any): unknown[] {
  const out: unknown[] = [];
  c.as_ordered((it: any) => {
    it.first();
    while (it.index() !== null) {
      const values: unknown[] = [];
      it.as_item((e: any) => e.as_scalar((s: any) => values.push(s.get()) && s));
      out.push([it.index(), values[0]]);
      it.next();
    }
  });
  return out;
}

## CNT-01 · Elements by index: written through a builder's container, read through a data instance's; reads never raise

In [ ]:
let [card] = store.transaction(() => {
  let card = store.Card((c: any) => (
    c
      .tags((p: any) => p.as_value((a: any) => a.as_container(
        (t: any) => t.at(0n).as_scalar("a") && t.at(2n).as_scalar("c"))))
      .phones((p: any) => p.as_value((a: any) => a.as_container(
        (d: any) => d.at("home").as_object((r: any) => r.number("1")))))
      .grid((p: any) => p.as_value((a: any) => a.as_container((g: any) => g.at([1n, 2n]).as_scalar(3.5))))
  ));
  return [card];
});
store.transaction(() => {
  {
    const [tags, phones, grid] = [container(card.tags), container(card.phones), container(card.grid)];
    assert(tags.as_schema() === Tags && same([tags.length, phones.length, grid.length], [3, 1, 6])); // dense: every index up to the last
    assert(equal(scalars(tags), [[0n, "a"], [2n, "c"]]) && valueOfAny(tags.at(1n)) === null); // an index with nothing reads as empty
    assert(equal(walk(tags), [[0n, "a"], [1n, null], [2n, "c"]])); // an iterator visits every position of a dense container
    assert(equal(walk(grid), [[[0n, 0n], null], [[0n, 1n], null], [[0n, 2n], null], [[1n, 0n], null], [[1n, 1n], null], [[1n, 2n], 3.5]]));
    const seen: unknown[] = [];
    phones.as_dict((d: Map<unknown, unknown>) => seen.push([...d.keys()].sort()));
    tags.as_dict(() => seen.push("not a dict")); // not called: tags is dense
    phones.as_ordered(() => seen.push("not ordered")); // not called: phones is a dict
    assert(equal(seen, [["home"]]));
    phones.at("home").as_object((r: any) => seen.push(`${r.number}`) && r);
    assert(seen.at(-1) === "1");
    assert(container(store.Card((c: any) => c).tags) === null); // not set
    card.name.as_value((a: any) => a.as_container((c: any) => seen.push([c.as_schema(), c.length, c.at(0n).as_schema()])));
    assert(equal(seen.at(-1), [null, 0, null])); // a value of another kind: an empty visitor, read as an empty container
    card.name.as_value((a: any) => a.as_container((c: any) => c.at(0n).as_container((x: any) => seen.push(x.as_schema()))));
    assert(seen.at(-1) === null);
    const empty: unknown[] = [];
    card.name.as_value((a: any) => a.as_container((c: any) => c.as_entries((pairs: Iterable<unknown>) => empty.push([...pairs]))
      .as_ordered(() => empty.push("x")).as_dict(() => empty.push("x"))));
    assert(equal(empty, [[]]));
    raises(ReadOnlyError, () => tags.remove(0n), "the container is read-only; write through a builder");
    raises(ReadOnlyError, () => tags.at(0n).as_scalar("z"), "the container[0] is read-only; write through a builder");
    raises(ReadOnlyError, () => card.tags.as_value((a: any) => a.as_container(tags)), "property 'tags' is read-only; write through a builder");
    raises(ReadOnlyError, () => store.Card((c: any) => c.name((p: any) => p.as_value((a: any) => a.as_container(tags)))),
      "property 'name' does not hold a container");
    assert(repr(tags) === "<container of 2>");
    // A builder edits a copy, which it reads back; an update writes it, a clone copies it.
    const edited = store.Card(card, (c: any) => c.tags((p: any) => p.as_value((a: any) => a.as_container((t: any) => {
      t.remove(2n).at(1n).as_scalar("b");
      assert(t.length === 2);
    }))));
    assert(edited === card && equal(scalars(container(card.tags)), [[0n, "a"], [1n, "b"]]));
    const copy = store.Card(card);
    assert(equal(scalars(container(copy.tags)), [[0n, "a"], [1n, "b"]]) && container(copy.tags) !== container(card.tags));
    // A container given as a value is written as it is; one of another schema is refused.
    const given = store.Card((c: any) => c.tags((p: any) => p.as_value((a: any) => a.as_container(container(card.tags)))));
    assert(equal(scalars(container(given.tags)), [[0n, "a"], [1n, "b"]]));
    raises(TypeError, () => store.Card((c: any) => c.tags((p: any) => p.as_value((a: any) => a.as_container(
      container(card.grid))))),
      "property 'tags' takes a container of its schema or a Spec, got _ContainerData");
    assert(equal(scalars(container(store.Card((c: any) => c.tags(container(card.tags))).tags)), [[0n, "a"], [1n, "b"]])); // as a property's value
    const twice = store.Card((c: any) => (
      c
        .tags((p: any) => p.as_value((a: any) => a.as_container((t: any) => t.at(0n).as_scalar("a"))))
        .tags((p: any) => p.as_value((a: any) => a.as_container((t: any) => t.at(1n).as_scalar("b"))))
    ));
    assert(equal(scalars(container(twice.tags)), [[0n, "a"], [1n, "b"]])); // a builder reads back, and adds to, what it has written
    // Containers nest: a container's element may be a container.
    const nested = store.Card((c: any) => c.rows((p: any) => p.as_value((a: any) => a.as_container(
      (r: any) => r.at(1n).as_container((t: any) => t.at(0n).as_scalar("x"))))));
    const inner: unknown[] = [];
    container(nested.rows).at(1n).as_container((t: any) => inner.push(scalars(t)));
    assert(equal(inner, [[[0n, "x"]]]) && container(nested.rows).length === 2);
  }
});

## CNT-02 · Iterators: positions before the first, at each, and past the last; a builder's extends a container and inserts into one of int indices

In [ ]:
{
  store.transaction(() => {
    const sparse = store.Card((c: any) => c.sparse((p: any) => p.as_value((a: any) => a.as_container(
      (c: any) => c.at(10n).as_scalar("j") && c.at(11n).as_scalar("k") && c.at(40n).as_scalar("z")))));
    assert(equal(walk(container(sparse.sparse)), [[10n, "j"], [11n, "k"], [40n, "z"]]) && container(sparse.sparse).length === 3);
    const moves: unknown[] = [];
    container(sparse.sparse).as_ordered((it: any) => moves.push(it.index(), it.first().index(), it.next().index(), it.last().index(),
      it.next().index(), it.next().index(), it.prev().index(), it.first().prev().index(), it.prev().index(), String(it)));
    assert(equal(moves, [null, 10n, 11n, 40n, null, null, 40n, null, null, "<iterator at None>"]));
    const calls: unknown[] = [];
    container(sparse.sparse).as_ordered((it: any) => it.last().next().as_item((e: unknown) => calls.push(e)));
    assert(calls.length === 0); // past the end of a read-only container: not called
    raises(ReadOnlyError, () => container(sparse.sparse).as_ordered((it: any) => it.first().prepend("x")),
      "the container is read-only; write through a builder");
    // A builder's iterator edits a card's container.
    store.Card(sparse, (c: any) => c.sparse((p: any) => p.as_value((a: any) => a.as_container((c: any) => c.as_ordered(
      (it: any) => it.last().next().as_item((e: any) => e.as_scalar("past")).first().prev().as_item(
        (e: any) => e.as_scalar("before")))))));
    assert(equal(walk(container(sparse.sparse)), [[10n, "before"], [11n, "j"], [12n, "k"], [41n, "z"], [42n, "past"]]));
    store.Card(sparse, (c: any) => c.sparse((p: any) => p.as_value((a: any) => a.as_container((c: any) => c.as_ordered(
      (it: any) => it.first().append("after").prepend((e: any) => e.as_scalar("in")))))));
    assert(equal(walk(container(sparse.sparse)), [[10n, "before"], [11n, "in"], [12n, "after"], [13n, "j"], [14n, "k"], [43n, "z"], [44n, "past"]]));
    const fresh = store.Card((c: any) => c.tags((p: any) => p.as_value((a: any) => a.as_container((c: any) => c.as_ordered(
      (it: any) => it.as_item((e: any) => e.as_scalar("first")).next().as_item((e: any) => e.as_scalar("second")))))));
    assert(equal(walk(container(fresh.tags)), [[0n, "first"], [1n, "second"]])); // an empty container extended from its ends
    store.Card(fresh, (c: any) => c.tags((p: any) => p.as_value((a: any) => a.as_container((c: any) => c.as_ordered(
      (it: any) => it.last().next().append("end").first().prev().prepend("start"))))));
    assert(equal(walk(container(fresh.tags)), [[0n, "start"], [1n, "first"], [2n, "second"], [3n, "end"]]));
    const blank = store.Card((c: any) => c.tags((p: any) => p.as_value((a: any) => a.as_container(
      (c: any) => c.as_ordered((it: any) => it.append("only"))))));
    assert(equal(walk(container(blank.tags)), [[0n, "only"]]));
    const ends = store.Card((c: any) => c.tags((p: any) => p.as_value((a: any) => a.as_container((c: any) => c.as_ordered(
      (it: any) => it.first().as_item((e: any) => e.as_scalar("a")).last().next().prepend("z").first().prev().append("y"))))));
    assert(equal(walk(container(ends.tags)), [[0n, "y"], [1n, "a"], [2n, "z"]])); // at the ends, of an empty container and of one with elements
    const lone = store.Card((c: any) => c.tags((p: any) => p.as_value((a: any) => a.as_container(
      (c: any) => c.as_ordered((it: any) => it.prepend("p"))))));
    assert(equal(walk(container(lone.tags)), [[0n, "p"]]));
    const gridded = store.Card(card, (c: any) => c.grid((p: any) => p.as_value((a: any) => a.as_container(
      (g: any) => g.as_ordered((it: any) => it.first().as_item((e: any) => e.as_scalar(0.5)))))));
    assert(equal(walk(container(gridded.grid))[0], [[0n, 0n], 0.5]));
    raises(ValueError, () => store.Card(card, (c: any) => c.grid((p: any) => p.as_value((a: any) => a.as_container(
      (c: any) => c.as_ordered((it: any) => it.first().append(1.0)))))),
      "property 'grid': only a dense or sparse container of int indices has positions to insert at; write through at(index)");
  });
}

## CNT-03 · Value objects in containers belong to the object holding the container: edited in place, copied by a clone, removed with their element

In [ ]:
/** Box, holding a kit: a value object of no name holding a container. */
function boxes(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("cnt.boxes/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Box: S.OfObject })
    .spec([
      (_: any) => _.Box.name("Box").ref(true).properties({ kit: S.OfObject((o: any) => o.properties({ parts: txn.Tags })) }),
    ])]))]);
}

{
  store.transaction(boxes);
  store.transaction(() => {
    const owner = store.Card((c: any) => c.phones((p: any) => p.as_value((a: any) => a.as_container(
      (d: any) => d.at("home").as_object((r: any) => r.number("1").label("h"))
      && d.at("work").as_object((r: any) => r.number("2"))))));
    const held = new Map<unknown, any>();
    container(owner.phones).as_dict((d: Map<unknown, any>) => d.forEach((e, k) => held.set(k, e)));
    const home: any[] = [];
    held.get("home").as_object((r: any) => home.push(r) && r);
    assert(Proxies.owner_of(home[0]) === owner && `${home[0].label}` === "h");
    store.Card(owner, (c: any) => c.phones((p: any) => p.as_value((a: any) => a.as_container(
      (d: any) => d.at("home").as_object((r: any) => r.label("x"))))));
    const again: any[] = [];
    container(owner.phones).at("home").as_object((r: any) => again.push(r) && r);
    assert(again[0] === home[0] && `${home[0].label}` === "x" && `${home[0].number}` === "1"); // edited in place
    const twin = store.Card(owner);
    const copied: any[] = [];
    container(twin.phones).at("home").as_object((r: any) => copied.push(r) && r);
    assert(copied[0] !== home[0] && Proxies.owner_of(copied[0]) === twin && `${copied[0].number}` === "1");
    store.Card(owner, (c: any) => c.phones((p: any) => p.as_value((a: any) => a.as_container(
      (d: any) => d.remove("home")))));
    assert(container(owner.phones).length === 1 && Proxies.owner_of(home[0]) === owner);
    const box = store.Box((b: any) => b.kit((p: any) => p.as_object((k: any) => k.parts((p: any) => p.as_value((a: any) => a.as_container(
      (t: any) => t.at(0n).as_scalar("bolt")))))));
    const twinBox = store.Box(box);
    const parts: unknown[] = [];
    twinBox.kit.as_object((k: any) => {
      k.parts.as_value((a: any) => a.as_container((t: any) => parts.push(scalars(t))));
      return k;
    });
    assert(equal(parts, [[[0n, "bolt"]]])); // copied with the value object that holds it
    const moved = store.Card((c: any) => c.tags((p: any) => p.as_value((a: any) => a.as_container(
      (t: any) => t.at(0n).as_scalar("a")))));
    store.Card(moved, (c: any) => c.tags((p: any) => p.as_value((a: any) => a.as_container(
      (t: any) => t.as_ordered((it: any) => it.first().prepend("z"))))));
    assert(equal(walk(container(moved.tags)), [[0n, "z"], [1n, "a"]]));
  });
}

## CNT-04 · Snapshots: a dense container as an array, nested or flat for several indices, a sparse one as blocks, a dict by text as a mapping, entries otherwise; read back, and over what a store holds

In [ ]:
/** Note, holding containers of other indices: a dict by bytes, a dense container by text, a sparse one by cell, and a
 * dict by a value object of text and int. */
function noting(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("cnt.noting/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Note: S.OfObject })
    .spec([
      (_: any) => _.Note.name("Note").ref(true).properties({
        b: S.OfContainer((c: any) => c.index([S.OfBytes]).element([S.OfInt])),
        s: S.OfContainer((c: any) => c.index([S.OfStr]).element([S.OfInt]).layout("dense")),
        k: S.OfContainer((c: any) => c.index([txn.Cell]).element([S.OfInt]).layout("sparse")),
        m: S.OfContainer((c: any) => c.index([S.OfObject((o: any) => o.properties({ k: S.OfStr, n: S.OfInt }))]).element([
          S.OfInt])),
      }),
    ])]))]);
}

/** Choice, picking a union of no name whose branch holds a container. */
function choosing(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("cnt.choosing/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Choice: S.OfObject })
    .spec([
      (_: any) => _.Choice.name("Choice").ref(true).properties({ pick: S.OfUnion((n: any) => n.branches([
        (t: any) => t.name("tags").schema([txn.Tags]),
        (t: any) => t.name("word").schema([S.OfStr]),
      ])) }),
    ])]))]);
}

/** Sets, a set of entries holding containers, and Marks, at both its ends. */
function marking(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("cnt.marking/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Sets: S.OfRelation, Marks: S.OfObject })
    .spec([
      (_: any) => _.Sets.name("Sets").links([(t: any) => t.name("a"), (t: any) => t.name("b")]).properties({
        tags: txn.Tags, grid: txn.Grid }).uniques([[]]),
      (_: any) => _.Marks.name("Marks").ref(true).adjacencies({
        sets: (a: any) => a.relation([_.Sets]).me("a"),
        set_by: (a: any) => a.relation([_.Sets]).me("b"),
      }),
    ])]))]);
}

store.transaction(() => {
  card = store.Card((c: any) => (
    c
      .tags((p: any) => p.as_value((a: any) => a.as_container(
        (t: any) => t.at(0n).as_scalar("a") && t.at(2n).as_scalar("c"))))
      .phones((p: any) => p.as_value((a: any) => a.as_container(
        (d: any) => d.at("home").as_object((r: any) => r.number("1")))))
      .grid((p: any) => p.as_value((a: any) => a.as_container(
        (g: any) => g.at([1n, 2n]).as_scalar(3.5) && g.at([0n, 0n]).as_scalar(0.25))))
      .sparse((p: any) => p.as_value((a: any) => a.as_container(
        (s: any) => s.at(3n).as_scalar("x") && s.at(4n).as_scalar("y") && s.at(9n).as_scalar("z"))))
      .rows((p: any) => p.as_value((a: any) => a.as_container(
        (r: any) => r.at(0n).as_container((t: any) => t.at(1n).as_scalar("n")))))
  ));
});
const written = root_of(Plain.ToPlain(store).OfObject(Card, card));
assert(container(card.grid).length === 6); // the box its indices span
assert(equal(written, map({ tags: ["a", null, "c"], phones: { home: { number: "1" } }, grid: [[0.25, null, null], [null, null, 3.5]],
  sparse: [{ at: 3n, items: ["x", "y"] }, { at: 9n, items: ["z"] }], rows: [[null, "n"]] })));
const flat = root_of(Plain.ToPlain(store, { flat: true }).OfObject(Card, card));
assert(equal(flat.get("grid"), map({ shape: [2n, 3n], items: [0.25, null, null, null, null, 3.5] })));
assert(JSON.ToJSON(store, { flat: true }).OfObject(Card, card).includes('"value": {"shape": [2, 3]')
  && YAML.ToYAML(store, { flat: true }).OfObject(Card, card).includes("shape:"));
const other: any = new B.Store(); // another store of the same schemas
Plain.FromPlain(other).Snapshot(Plain.ToPlain(store).Snapshot()); // store's schemas, as they are now: other's own
for (const snapshot of [Plain.ToPlain(store).OfObject(Card, card), Plain.ToPlain(store, { flat: true }).OfObject(Card, card)]) {
  const back = Plain.FromPlain(other).OfObject(other.Card, snapshot) as any; // the loader takes either form
  assert(equal(root_of(Plain.ToPlain(other).OfObject(other.Card, back)), written));
}
const again: any = Plain.FromPlain(store).OfObject(Card, Plain.ToPlain(store).OfObject(Card, card));
assert(again !== card && equal(root_of(Plain.ToPlain(store).OfObject(Card, again)), written)); // reloading a counted object makes a new one
// Any other index is written as entry-like elements: a dict by bytes, a dense container by text, a sparse one by text.
store.transaction(noting);
const Note = store.registered("Note");
const [note] = store.transaction(() => {
  const note = store.Note((n: any) => (
    n
      .b((p: any) => p.as_value((a: any) => a.as_container((c: any) => c.at(new Uint8Array([1])).as_scalar(1n))))
      .s((p: any) => p.as_value((a: any) => a.as_container((c: any) => c.at("b").as_scalar(2n) && c.at("a").as_scalar(1n))))
      .k((p: any) => p.as_value((a: any) => a.as_container(
        (c: any) => c.at([0n, 1n]).as_scalar(1n) && c.at([0n, 2n]).as_scalar(2n) && c.at([3n, 0n]).as_scalar(3n))))
      .m((p: any) => p.as_value((a: any) => a.as_container((c: any) => c.at(["x", 1n]).as_scalar(5n))))
  ));
  return [note];
});
const noted = root_of(Plain.ToPlain(store).OfObject(Note, note));
assert(equal(noted, map({ b: [{ index: "AQ==", element: 1n }], s: [{ index: "a", element: 1n }, { index: "b", element: 2n }],
  k: [{ at: { row: 0n, col: 1n }, items: [1n, 2n] }, { at: { row: 3n, col: 0n }, items: [3n] }],
  m: [{ index: { k: "x", n: 1n }, element: 5n }] })));
Plain.FromPlain(other).Snapshot(Plain.ToPlain(store).Snapshot()); // store's schemas, as they are now: other's own
assert(equal(root_of(Plain.ToPlain(other).OfObject(other.Note, Plain.FromPlain(other).OfObject(other.Note, Plain.ToPlain(store).OfObject(Note, note)) as any)), noted));
// A union's branch, or an intersection's part, may hold a container.
store.transaction(choosing);
const Choice = store.registered("Choice");
Plain.FromPlain(other).Snapshot(Plain.ToPlain(store).Snapshot()); // store's schemas, as they are now: other's own
const [choice] = store.transaction(() => {
  const choice = store.Choice((c: any) => c.pick((p: any) => p.as_value((a: any) => a.as_union((u: any) => u.tags((t: any) => t.at(1n).as_scalar("q"))))));
  return [choice];
});
const picked = root_of(Plain.ToPlain(store).OfObject(Choice, choice));
assert(equal(picked, map({ pick: { tags: [null, "q"] } })));
assert(equal(root_of(Plain.ToPlain(other).OfObject(other.Choice, Plain.FromPlain(other).OfObject(other.Choice, Plain.ToPlain(store).OfObject(Choice, choice)) as any)), picked));
store.transaction(() => {
  store.Choice(choice, (c: any) => c.pick((p: any) => p.as_value((a: any) => a.as_union((u: any) => u.tags((t: any) => t.at(2n).as_scalar("r"))))));
});
assert(equal(root_of(Plain.ToPlain(store).OfObject(Choice, choice)), map({ pick: { tags: [null, "q", "r"] } }))); // its container edited
const [sameBranch] = store.transaction(() => {
  const sameBranch = store.Choice((c: any) => c.pick((p: any) => p.as_value((a: any) => a.as_union(
    (u: any) => u.tags((t: any) => t.at(0n).as_scalar("p")).tags((t: any) => t.at(1n).as_scalar("q")))))); // read back, then written again
  return [sameBranch];
});
assert(equal(root_of(Plain.ToPlain(store).OfObject(Choice, sameBranch)), map({ pick: { tags: ["p", "q"] } })));
const [givenBranch] = store.transaction(() => {
  const givenBranch = store.Choice((c: any) => c.pick((p: any) => p.as_value((a: any) => a.as_union((u: any) => u.tags(container(card.tags))))));
  return [givenBranch];
});
assert(equal(root_of(Plain.ToPlain(store).OfObject(Choice, givenBranch)), map({ pick: { tags: ["a", null, "c"] } }))); // a container given
store.transaction(() => {
  raises(TypeError, () => store.Choice((c: any) => c.pick((p: any) => p.as_value((a: any) => a.as_union((u: any) => u.tags(1n))))),
    "branch 'tags' takes a container of its schema or a Spec, got int");
});
raises(ReadOnlyError, () => container(card.tags)((t: unknown) => t), "the container is read-only; write through a builder");
// An entry's property may hold a container: entries equal by their containers are one, in a set.
store.transaction(marking);
const Marks = store.registered("Marks");
Plain.FromPlain(other).Snapshot(Plain.ToPlain(store).Snapshot()); // store's schemas, as they are now: other's own
const [target, marks] = store.transaction(() => {
  const target = store.Marks((m: any) => m);
  const marks = store.Marks((m: any) => m.sets((adj: any) => (
    adj
      .insert((e: any) => e.b(target)
        .tags((p: any) => p.as_value((a: any) => a.as_container(
          (t: any) => t.at(0n).as_scalar("a") && t.at(1n).as_scalar("b"))))
        .grid((p: any) => p.as_value((a: any) => a.as_container((g: any) => g.at([0n, 1n]).as_scalar(1.5)))))
      .insert((e: any) => e.b(target)
        .tags((p: any) => p.as_value((a: any) => a.as_container(
          (t: any) => t.at(0n).as_scalar("a") && t.at(1n).as_scalar("b"))))
        .grid((p: any) => p.as_value((a: any) => a.as_container((g: any) => g.at([0n, 1n]).as_scalar(1.5)))))
  )));
  return [target, marks];
});
const row = { b: ref(1), tags: ["a", "b"], grid: [[null, 1.5]] };
assert(marks.sets.length === 1 && equal(root_of(Plain.ToPlain(store).Reachable(Marks, marks)), map({ sets: [row] })));
const doubled = map({ root: { n: 0n }, objects: [listed("Marks", 0, { sets: [row, row] }), listed("Marks", 1)] });
assert((Plain.FromPlain(other).OfObject(other.Marks, document(doubled)) as any).sets.length === 1); // read once, by value
assert(container(note.s).length === 2); // a dense container by text: the elements present
const emptied = Plain.FromPlain(other).OfObject(other.Note, document(single("Note", { b: [{ index: "AQ==" }], s: [] }))) as any;
assert(equal(root_of(Plain.ToPlain(other).OfObject(other.Note, emptied)), map({ b: [], s: [] }))); // an element absent; empty containers
const branch: any[] = [];
choice.pick.as_value((a: any) => a.as_union((u: any) => branch.push(u.tags) && u));
store.transaction(() => {
  store.Choice((c: any) => c).pick.as_value((a: any) => a.as_union((u: any) => branch.push(u.tags) && u));
});
assert(branch[0].length === 3 && branch[1].as_schema() === null); // a union value not set: its branch reads as empty

## CNT-05 · Malformed containers in snapshots are refused, with a path

In [ ]:
/** Deep, a value object holding a cell; Odd, a container by Deep, whose index is neither ordered nor hashable; and Rare,
 * holding an Odd. */
function rare(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("cnt.rare/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Deep: S.OfObject, Odd: S.OfContainer, Rare: S.OfObject })
    .spec([
      (_: any) => _.Deep.name("Deep").properties({ cell: txn.Cell }),
      (_: any) => _.Odd.name("Odd").index([_.Deep]).element([S.OfInt]), // entries: its index is neither ordered nor hashable
      (_: any) => _.Rare.name("Rare").ref(true).properties({ odd: _.Odd }),
    ])]))]);
}

const at = '$.mutations[0].upsert.exists[0].spec[0]["0"][0].as_property.mutations[0].set.value'; // its one member's value
{
  const refused = (name: string, value: unknown, message: string) =>
    raises(DecodeError, () => Plain.FromPlain(store).OfObject(Card, document(single("Card", { [name]: value }))), message);
  refused("tags", { a: 1n }, `${at}: a dense container must be a list, got dict`);
  refused("tags", [1n], `${at}[0]: expected str, got int`);
  refused("grid", [[1.0], 2.0], `${at}[1]: a container of 2 indices nests 2 lists`);
  refused("grid", { shape: [2n], items: [1.0] }, `${at}: a flat container's shape is 2 sizes, and its items as many as they make`);
  refused("grid", { shape: [1n, 2n], items: [1.0] }, `${at}: a flat container's shape is 2 sizes`);
  refused("grid", { shape: "2, 3", items: [] }, `${at}: a flat container's shape is 2 sizes`);
  refused("grid", { shape: [1n, 1n], items: [true] }, `${at}.items[0]: expected float, got bool`);
  refused("sparse", { at: 1n }, `${at}: a sparse container must be a list of blocks, got dict`);
  refused("sparse", [{ at: 1n }], `${at}[0]: a block is {'at': index, 'items': [element, ...]}`);
  refused("sparse", [{ at: "1", items: [] }], `${at}[0].at: expected int, got str`);
  refused("phones", [], `${at}: a container keyed by text must be a mapping, got list`);
  refused("phones", { home: 1n }, `${at}.home: a value object must be a mapping, got int`);
  refused("phones", { "home office": 1n }, `${at}["home office"]: a value object must be a mapping, got int`); // not a name: quoted
  raises(DecodeError, () => Plain.FromPlain(store).OfObject(Note, document(single("Note", { m: {} }))),
    `${at}: a container must be a list, got dict`);
  raises(DecodeError, () => Plain.FromPlain(store).OfObject(Note, document(single("Note", { m: [{ element: 1n }] }))),
    `${at}[0]: an entry of a container is {'index': index, 'element': element}`);
  raises(DecodeError, () => Plain.FromPlain(store).OfObject(Note, document(single("Note", { k: [{ at: { row: 1n }, items: [1n] }] }))),
    `${at}[0].at: an index has every property of its value object; ['col'] are not set`);
}
store.transaction(rare);
const Deep = store.registered("Deep");
const Rare = store.registered("Rare");
raises(DecodeError, () => Plain.FromPlain(store).OfObject(Rare, document(single("Rare", { odd: [{ index: {}, element: 1n }] }))),
  `${at}[0].index: a container whose index is neither ordered nor hashable is not read from snapshots yet`);

## CNT-06 · An index is checked against the index's schema: a scalar of its host type, or a tuple of a value object index's properties; a dense container's is not negative

In [ ]:
/** Keyed, holding a Deep and an Odd: a container by Deep. */
function keying(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("cnt.keying/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Keyed: S.OfObject })
    .spec([
      (_: any) => _.Keyed.name("Keyed").ref(true).properties({ key: txn.Deep, odd: txn.Odd }),
    ])]))]);
}

{
  store.transaction(keying);
  const Keyed = store.registered("Keyed");
  store.transaction(() => {
    raises(TypeError, () => store.Card((c: any) => c.tags((p: any) => p.as_value((a: any) => a.as_container(
      (c: any) => c.at("0"))))), "property 'tags': an index is a int, got str");
    raises(TypeError, () => store.Card((c: any) => c.tags((p: any) => p.as_value((a: any) => a.as_container(
      (c: any) => c.at(true))))), "property 'tags': an index is a int, got bool");
    raises(TypeError, () => store.Card((c: any) => c.grid((p: any) => p.as_value((a: any) => a.as_container(
      (c: any) => c.at([1n]))))),
      "property 'grid': an index is a tuple of int, int, got [1]");
    raises(TypeError, () => store.Card((c: any) => c.grid((p: any) => p.as_value((a: any) => a.as_container(
      (c: any) => c.at([1n, "2"]))))),
      "property 'grid': an index is a tuple of int, int, got [1, '2']");
    raises(ValueError, () => store.Card((c: any) => c.tags((p: any) => p.as_value((a: any) => a.as_container(
      (c: any) => c.at(-1n))))),
      "property 'tags': a dense container's index is not negative, got -1");
    raises(ValueError, () => store.Card((c: any) => c.grid((p: any) => p.as_value((a: any) => a.as_container(
      (c: any) => c.at([0n, -1n]))))),
      "property 'grid': a dense container's index is not negative, got [0, -1]");
    store.builder("Card").sparse((p: any) => p.as_value((a: any) => a.as_container(
      (c: any) => c.at(-5n).as_scalar("ok")))); // sparse: any int
    const deep: any[] = [];
    store.builder("Rare").odd((p: any) => p.as_value((a: any) => a.as_container((c: any) => deep.push(c))));
    raises(TypeError, () => deep[0].at(1n), "property 'odd': an index is a value of its index type, got int");
    // An index of a value object that is neither ordered nor hashable: given as a value object, written as one.
    let keyed = store.Keyed((k: any) => k.key((p: any) => p.as_object((k: any) => k.cell((q: any) => q.as_object((c: any) => c.row(1n).col(2n))))));
    const key: any[] = [];
    keyed.key.as_object((k: any) => key.push(k) && k);
    keyed = store.Keyed(keyed, (k: any) => k.odd((p: any) => p.as_value((a: any) => a.as_container(
      (c: any) => c.at(key[0]).as_scalar(7n)))));
    assert(equal(root_of(Plain.ToPlain(store).OfObject(Keyed, keyed)).get("odd"), map([{ index: { cell: { row: 1n, col: 2n } }, element: 7n }])));
  });
}

## CNT-07 · Validation and comparison: each element checked at its index; containers compared by index and element, by value in uniqueness; different index or element schemas incomparable

In [ ]:
/** Shelf, holding containers of text, of lines by text, of floats by cell, of Words, and of ints by Deep, and Labels,
 * whose entries hold some too. */
function shelving(txn: any): any {
  const S = txn.Schemas;
  return txn.uid("cnt.shelving/1").mutations([(mu: any) => mu.upsert((u: any) => u.exists([(p: any) => p
    .symbols({ Line: S.OfObject, Words: S.OfContainer, Table: S.OfContainer, ByDeep: S.OfContainer,
      Labels: S.OfRelation, Shelf: S.OfObject })
    .spec([
      (_: any) => _.Line.name("Line").properties({ number: S.OfStr }),
      (_: any) => _.Words.name("Words").index([S.OfInt]).element([S.OfStr]),
      (_: any) => _.Table.name("Table").index([txn.Cell]).element([S.OfFloat]),
      (_: any) => _.ByDeep.name("ByDeep").index([txn.Deep]).element([S.OfInt]),
      (_: any) => _.Labels.name("Labels").links([(t: any) => t.name("shelf"), (t: any) => t.name("label")]).properties({
        words: _.Words, deep: _.ByDeep, table: _.Table, line: _.Line }).uniques("label"),
      (_: any) => _.Shelf.name("Shelf").ref(true).properties({
        words: _.Words,
        book: S.OfContainer((c: any) => c.index([S.OfStr]).element([_.Line])),
        table: _.Table,
        nest: S.OfContainer((c: any) => c.index([S.OfInt]).element([_.Words])),
        deep: _.ByDeep,
        key: txn.Deep,
      }).adjacencies({
        labels: (a: any) => a.relation([_.Labels]).me("shelf"),
        labelled: (a: any) => a.relation([_.Labels]).me("label"),
      }),
    ])]))]);
}

{
  store.transaction(shelving);
  const [Line, Words, Table, Shelf] = ["Line", "Words", "Table", "Shelf"].map((name) => store.registered(name));
  const held = (read: any): any => {
    const found: any[] = [];
    read.as_object((k: any) => found.push(k) && k);
    return found[0];
  };
  const compare = (x: any, y: any) =>
    new Comparison.OfObject(Shelf, x, { value: true }).compare(new Comparison.OfObject(Shelf, y, { value: true }));
  const [a, unique] = store.transaction(() => {
    const a = store.Shelf((s: any) => (
      s
        .words((p: any) => p.as_value((v: any) => v.as_container((c: any) => c.at(0n).as_scalar("a") && c.at(2n).as_scalar("c"))))
        .book((p: any) => p.as_value((v: any) => v.as_container(
          (c: any) => c.at("home").as_object((r: any) => r.number("1")) && c.at("work").as_object((r: any) => r.number("2")))))
        .table((p: any) => p.as_value((v: any) => v.as_container((c: any) => c.at([0n, 1n]).as_scalar(0.5))))
        .nest((p: any) => p.as_value((v: any) => v.as_container((c: any) => c.at(0n).as_container((w: any) => w.at(1n).as_scalar("x")))))
        .key((p: any) => p.as_object((k: any) => k.cell((q: any) => q.as_object((c: any) => c.row(1n).col(0n)))))
    ));
    store.Shelf(a, (s: any) => s.deep((p: any) => p.as_value((v: any) => v.as_container((c: any) => c.at(held(a.key)).as_scalar(1n)))));
    const reordered = store.Shelf((s: any) => (
      s
        .words((p: any) => p.as_value((v: any) => v.as_container((c: any) => c.at(0n).as_scalar("a") && c.at(2n).as_scalar("c"))))
        .book((p: any) => p.as_value((v: any) => v.as_container(
          (c: any) => c.at("work").as_object((r: any) => r.number("2")) && c.at("home").as_object((r: any) => r.number("1")))))
        .table((p: any) => p.as_value((v: any) => v.as_container((c: any) => c.at([0n, 1n]).as_scalar(0.5))))
        .nest((p: any) => p.as_value((v: any) => v.as_container((c: any) => c.at(0n).as_container((w: any) => w.at(1n).as_scalar("x")))))
        .key((p: any) => p.as_object((k: any) => k.cell((q: any) => q.as_object((c: any) => c.row(1n).col(0n)))))
    ));
    store.Shelf(reordered, (s: any) => s.deep((p: any) => p.as_value((v: any) => v.as_container(
      (c: any) => c.at(held(reordered.key)).as_scalar(1n)))));
    assert(compare(a, reordered) === 0); // a dict's order is not compared
    const moved = store.Shelf((s: any) => (
      s
        .words((p: any) => p.as_value((v: any) => v.as_container((c: any) => c.at(0n).as_scalar("a") && c.at(1n).as_scalar("c"))))
        .book((p: any) => p.as_value((v: any) => v.as_container(
          (c: any) => c.at("home").as_object((r: any) => r.number("1")) && c.at("work").as_object((r: any) => r.number("2")))))
        .table((p: any) => p.as_value((v: any) => v.as_container((c: any) => c.at([0n, 1n]).as_scalar(0.5))))
        .nest((p: any) => p.as_value((v: any) => v.as_container((c: any) => c.at(0n).as_container((w: any) => w.at(1n).as_scalar("x")))))
        .key((p: any) => p.as_object((k: any) => k.cell((q: any) => q.as_object((c: any) => c.row(1n).col(0n)))))
    ));
    store.Shelf(moved, (s: any) => s.deep((p: any) => p.as_value((v: any) => v.as_container((c: any) => c.at(held(moved.key)).as_scalar(1n)))));
    assert(compare(a, moved) === null); // an element at another index
    const changed = store.Shelf((s: any) => (
      s
        .words((p: any) => p.as_value((v: any) => v.as_container((c: any) => c.at(0n).as_scalar("a") && c.at(2n).as_scalar("x"))))
        .book((p: any) => p.as_value((v: any) => v.as_container(
          (c: any) => c.at("home").as_object((r: any) => r.number("1")) && c.at("work").as_object((r: any) => r.number("2")))))
        .table((p: any) => p.as_value((v: any) => v.as_container((c: any) => c.at([0n, 1n]).as_scalar(0.5))))
        .nest((p: any) => p.as_value((v: any) => v.as_container((c: any) => c.at(0n).as_container((w: any) => w.at(1n).as_scalar("x")))))
        .key((p: any) => p.as_object((k: any) => k.cell((q: any) => q.as_object((c: any) => c.row(1n).col(0n)))))
    ));
    store.Shelf(changed, (s: any) => s.deep((p: any) => p.as_value((v: any) => v.as_container(
      (c: any) => c.at(held(changed.key)).as_scalar(1n)))));
    assert(compare(a, changed) === null); // another element
    const fewer = store.Shelf((s: any) => (
      s
        .words((p: any) => p.as_value((v: any) => v.as_container((c: any) => c.at(0n).as_scalar("a"))))
        .book((p: any) => p.as_value((v: any) => v.as_container(
          (c: any) => c.at("home").as_object((r: any) => r.number("1")) && c.at("work").as_object((r: any) => r.number("2")))))
        .table((p: any) => p.as_value((v: any) => v.as_container((c: any) => c.at([0n, 1n]).as_scalar(0.5))))
        .nest((p: any) => p.as_value((v: any) => v.as_container((c: any) => c.at(0n).as_container((w: any) => w.at(1n).as_scalar("x")))))
        .key((p: any) => p.as_object((k: any) => k.cell((q: any) => q.as_object((c: any) => c.row(1n).col(0n)))))
    ));
    store.Shelf(fewer, (s: any) => s.deep((p: any) => p.as_value((v: any) => v.as_container((c: any) => c.at(held(fewer.key)).as_scalar(1n)))));
    assert(compare(a, fewer) === null); // fewer elements
    const rekeyed = store.Shelf((s: any) => (
      s
        .words((p: any) => p.as_value((v: any) => v.as_container((c: any) => c.at(0n).as_scalar("a") && c.at(2n).as_scalar("c"))))
        .book((p: any) => p.as_value((v: any) => v.as_container(
          (c: any) => c.at("home").as_object((r: any) => r.number("1")) && c.at("work").as_object((r: any) => r.number("2")))))
        .table((p: any) => p.as_value((v: any) => v.as_container((c: any) => c.at([0n, 1n]).as_scalar(0.5))))
        .nest((p: any) => p.as_value((v: any) => v.as_container((c: any) => c.at(0n).as_container((w: any) => w.at(1n).as_scalar("x")))))
        .key((p: any) => p.as_object((k: any) => k.cell((q: any) => q.as_object((c: any) => c.row(2n).col(0n)))))
    ));
    store.Shelf(rekeyed, (s: any) => s.deep((p: any) => p.as_value((v: any) => v.as_container(
      (c: any) => c.at(held(rekeyed.key)).as_scalar(1n)))));
    assert(compare(a, rekeyed) === null); // another value object index
    const words = container(a.words);
    const Other = store.Schemas.OfContainer((c: any) => c.index([Text]).element([Text])); // another index type: the builders' own
    const Counts = store.Schemas.OfContainer((c: any) => c.index([Int]).element([Int])); // another element type
    assert(new Comparison.OfContainer(Words, words).compare(new Comparison.OfContainer(Words, container(a.words))) === 0);
    assert(new Comparison.OfContainer(Words).compare(new Comparison.OfContainer(Words)) === 0); // absent, both
    assert(new Comparison.OfContainer(Words, words).compare(new Comparison.OfContainer(Words)) === null);
    assert(new Comparison.OfContainer(Words, words).compare(new Comparison.OfContainer(Other)) === null);
    assert(new Comparison.OfContainer(Words, words).compare(new Comparison.OfContainer(Counts)) === null);
    // Uniqueness compares containers by value: two entries of one shelf with equal containers name one label.
    const [b, c] = [0, 1].map(() => store.Shelf((s: any) => (
      s
        .words((p: any) => p.as_value((v: any) => v.as_container((c: any) => c)))
        .book((p: any) => p.as_value((v: any) => v.as_container((c: any) => c)))
        .table((p: any) => p.as_value((v: any) => v.as_container((c: any) => c.at([0n, 1n]).as_scalar(0.5))))
        .nest((p: any) => p.as_value((v: any) => v.as_container((c: any) => c.at(0n).as_container((w: any) => w.at(1n).as_scalar("x")))))
        .key((p: any) => p.as_object((k: any) => k.cell((q: any) => q.as_object((c: any) => c.row(1n).col(0n)))))
    )));
    for (const empty of [b, c]) {
      store.Shelf(empty, (s: any) => s.deep((p: any) => p.as_value((v: any) => v.as_container(
        (c: any) => c.at(held(empty.key)).as_scalar(1n)))));
    }
    store.Shelf(a, (s: any) => s.labels((r: any) => (
      r
        .insert((x: any) => x.label(b).words((p: any) => p.as_value((v: any) => v.as_container((c: any) => c.at(0n).as_scalar("w"))))
          .deep((p: any) => p.as_value((v: any) => v.as_container((c: any) => c.at(held(a.key)).as_scalar(2n))))
          .table((p: any) => p.as_value((v: any) => v.as_container((c: any) => c.at([0n, 1n]).as_scalar(0.5)))))
        .insert((x: any) => x.label(c).words((p: any) => p.as_value((v: any) => v.as_container((c: any) => c.at(0n).as_scalar("w"))))
          .deep((p: any) => p.as_value((v: any) => v.as_container((c: any) => c.at(held(a.key)).as_scalar(2n))))
          .table((p: any) => p.as_value((v: any) => v.as_container((c: any) => c.at([0n, 1n]).as_scalar(0.5)))))
        .insert((x: any) => x.label(c).words((p: any) => p.as_value((v: any) => v.as_container((c: any) => c.at(0n).as_scalar("v"))))
          .deep((p: any) => p.as_value((v: any) => v.as_container((c: any) => c.at(held(a.key)).as_scalar(2n))))
          .table((p: any) => p.as_value((v: any) => v.as_container((c: any) => c.at([0n, 1n]).as_scalar(0.5)))))
    )));
    const unique = "unique(label) violated: entries agreeing on ['deep', 'line', 'shelf', 'table', 'words'] differ on ['label']"; // no line set
    assert(equal(Validators.Validate(store).OfObject(Shelf, a), [unique]));
    return [a, unique];
  });
  // Validation checks each element, at its index: schemas retyped in place, by their builders.
  store.transaction(() => {
    store.Schemas.OfContainer(Words, (c: any) => c.element([Int]));
    store.Schemas.OfContainer(Table, (c: any) => c.element([Int]));
    store.Schemas.OfObject(Line, (o: any) => o.properties([(t: any) => t.name("number").schema([Int])]));
  });
  assert(equal(Validators.Validate(store).OfObject(Shelf, a), [
    "Shelf#0.words[0]: expected int, got str", "Shelf#0.words[2]: expected int, got str",
    "Shelf#0.book['home'].number: expected int, got str", "Shelf#0.book['work'].number: expected int, got str",
    "Shelf#0.table[[0, 1]]: expected int, got float", "Shelf#0.nest[0][1]: expected int, got str",
    "Shelf#0.labels[0].words[0]: expected int, got str", "Shelf#0.labels[0].table[[0, 1]]: expected int, got float",
    "Shelf#0.labels[1].words[0]: expected int, got str", "Shelf#0.labels[1].table[[0, 1]]: expected int, got float",
    "Shelf#0.labels[2].words[0]: expected int, got str", "Shelf#0.labels[2].table[[0, 1]]: expected int, got float",
    unique]));
}